# 04 — Plots & Visualization

Run `02_Run_Baselines.ipynb` and `03_Train_RL.ipynb` first.

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

from functions.baseline import compute_all_metrics

RESULTS_DIR = "../Results_Intraday"
SAVE_PLOTS = True

COLORS = {
    "QQQ Buy-and-Hold":     "#636EFA",
    "Equal-Weight Monthly": "#EF553B",
    "Inverse Volatility":   "#00CC96",
    "Momentum Top-20%":     "#AB63FA",
    "Supervised + MVO":     "#FFA15A",
    "RL Agent":             "#FF6692",
    "QQQ (matched)":        "#636EFA",
}

def get_color(name):
    for key, color in COLORS.items():
        if key in name:
            return color
    return "#19D3F3"

## 1. Load Data

In [4]:
# Baselines (same OOS period as RL)
bl_equity_path = f"{RESULTS_DIR}/equity_curves_oos.csv"
bl_metrics_path = f"{RESULTS_DIR}/performance_metrics_oos.csv"
if not os.path.exists(bl_equity_path):
    bl_equity_path = f"{RESULTS_DIR}/equity_curves_full.csv"
    bl_metrics_path = f"{RESULTS_DIR}/performance_metrics_full.csv"

bl_equity = pd.read_csv(bl_equity_path, index_col=0, parse_dates=True)
bl_metrics = pd.read_csv(bl_metrics_path, index_col=0)
print(f"Baselines loaded from: {bl_equity_path}")
print(f"Baselines: {list(bl_equity.columns)}")

# RL (stitched OOS equity)
rl_available = os.path.exists(f"{RESULTS_DIR}/rl_equity_oos.csv")
if rl_available:
    rl_equity = pd.read_csv(f"{RESULTS_DIR}/rl_equity_oos.csv", index_col=0, parse_dates=True)
    rl_returns = pd.read_csv(f"{RESULTS_DIR}/rl_daily_returns_oos.csv", index_col=0, parse_dates=True)
    rl_metrics = pd.read_csv(f"{RESULTS_DIR}/rl_performance_metrics.csv", index_col=0)
    print(f"RL OOS: {rl_equity.index[0]} → {rl_equity.index[-1]} ({len(rl_equity)} days)")
    display(rl_metrics)
else:
    print("RL not trained yet. Run 03_Train_RL.py first.")

Baselines loaded from: ../Results_Intraday/equity_curves_oos.csv
Baselines: ['QQQ Buy-and-Hold', 'Equal-Weight Monthly', 'Inverse Volatility', 'Momentum Top-20%', 'Supervised + MVO']
RL OOS: 2023-06-05 19:00:00 → 2026-02-04 19:00:00 (1274 days)


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Sharpe,Sortino,Calmar,N Days,Avg Daily Turnover (%)
RL Agent,17.4473,3.1670,9.1791,14.4099,0.4722,0.3450,0.0758,0.3925,0.5034,0.2198,1274.0,16.2113
QQQ,81.0828,12.4524,13.8379,18.3123,0.7618,0.8999,0.6119,0.9176,0.9102,0.6800,1274.0,NaN


## 2. RL Agent vs QQQ Buy-and-Hold (same period)

Recalculate QQQ B&H starting from the exact first day of the RL OOS test set,
so both curves start at 1.0 on the same date. This is the **fair** comparison.

In [5]:
if rl_available:
    # Recalculate QQQ equity from RL returns (already aligned)
    qqq_matched = (1 + rl_returns["QQQ"]).cumprod()
    rl_eq = (1 + rl_returns["RL Agent"]).cumprod()

    # Plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rl_eq.index, y=rl_eq, name="RL Agent (OOS)", mode="lines",
        line=dict(color=COLORS["RL Agent"], width=3.0)))
    fig.add_trace(go.Scatter(
        x=qqq_matched.index, y=qqq_matched, name="QQQ Buy-and-Hold", mode="lines",
        line=dict(color=COLORS["QQQ (matched)"], width=2.0, dash="dash"), opacity=0.8))

    fig.update_layout(
        title="Equity Curves — RL Agent vs QQQ (same OOS period, both start at $1)",
        xaxis_title="Date", yaxis_title="Portfolio Value ($1 start)",
        template="plotly_white", height=550,
        legend=dict(x=0.02, y=0.98), hovermode="x unified")
    if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_equity_rl_vs_qqq.html")
    fig.show()

    # Metrics table: RL vs QQQ on SAME period
    rl_eq_arr = np.array([1.0] + list(rl_eq.values))
    qqq_eq_arr = np.array([1.0] + list(qqq_matched.values))
    rl_m = compute_all_metrics(rl_eq_arr)
    qqq_m = compute_all_metrics(qqq_eq_arr)

    fair_compare = pd.DataFrame({"RL Agent": rl_m, "QQQ Buy-and-Hold": qqq_m}).T
    print(f"\nFair comparison ({rl_eq.index[0].date()} → {rl_eq.index[-1].date()}):")
    display(fair_compare[["ARC (%)", "ASD (%)", "Max Drawdown (%)", "Sharpe", "Sortino", "Calmar", "IR1", "IR2"]])


Fair comparison (2023-06-05 → 2026-02-04):


,ARC (%),ASD (%),Max Drawdown (%),Sharpe,Sortino,Calmar,IR1,IR2
RL Agent,3.1670,9.1791,14.4099,0.3925,0.5034,0.2198,0.3450,0.0758
QQQ Buy-and-Hold,12.4524,13.8379,18.3123,0.9176,0.9102,0.6800,0.8999,0.6119


## 3. All Strategies Equity Chart

All strategies on the same OOS period.

In [6]:
fig = go.Figure()

# Baselines
for col in bl_equity.columns:
    fig.add_trace(go.Scatter(
        x=bl_equity.index, y=bl_equity[col], name=col, mode="lines",
        line=dict(color=get_color(col), width=2.0 if "QQQ" in col else 1.2), opacity=0.7))

# RL Agent
if rl_available:
    fig.add_trace(go.Scatter(
        x=rl_equity.index, y=rl_equity["RL Agent"],
        name="RL Agent (OOS)", mode="lines",
        line=dict(color=COLORS["RL Agent"], width=3.0)))

fig.update_layout(
    title="Equity Curves — All Strategies",
    xaxis_title="Date", yaxis_title="Portfolio Value ($1 start)",
    template="plotly_white", height=600,
    legend=dict(x=0.02, y=0.98), hovermode="x unified")
if SAVE_PLOTS: fig.write_html(f"{RESULTS_DIR}/plot_equity_all.html")
fig.show()

## 4. Metrics Comparison — RL vs All Baselines

All strategies evaluated on the same OOS period for a fair comparison.

In [7]:
if rl_available:
    rl_row = rl_metrics.loc[["RL Agent"]]
    combined = pd.concat([bl_metrics, rl_row]).sort_values("IR2", ascending=False)
    print("All strategies (same OOS period):")
    display(combined[["ARC (%)", "ASD (%)", "Max Drawdown (%)", "Sharpe", "Sortino", "IR1", "IR2", "Avg Daily Turnover (%)"]])

All strategies (same OOS period):


,ARC (%),ASD (%),Max Drawdown (%),Sharpe,Sortino,IR1,IR2,Avg Daily Turnover (%)
QQQ Buy-and-Hold,11.0648,14.0958,22.7683,0.8153,0.7783,0.7850,0.3815,0.0000
Inverse Volatility,6.8317,12.4314,20.9505,0.5991,0.8074,0.5495,0.1792,0.8877
Equal-Weight Monthly,6.7794,12.3934,21.2045,0.5964,0.8029,0.5470,0.1749,0.1843
RL Agent,3.1670,9.1791,14.4099,0.3925,0.5034,0.3450,0.0758,16.2113
Momentum Top-20%,0.4974,14.4569,39.0483,0.1058,0.1429,0.0344,0.0004,32.5013
Supervised + MVO,-1.1749,14.6516,35.3268,-0.0050,-0.0068,0.0000,0.0000,51.7510


In [8]:
# 5. Snapshot current run into HP-tagged folders
#
# Set this tag before running the cell, e.g.:
# HP_TAG = "RL_1_HP02"
#
# This will:
# - copy the entire `Code` folder into `Code_ALL/<HP_TAG>/Code`
# - copy all non-directory files from `Results` into `Results/<HP_TAG>`
# - copy `training_log.txt` into `training_logs/training_log_<HP_TAG>.txt`

import os
import shutil
from pathlib import Path

# >>> SET THIS BEFORE RUNNING <<<
HP_TAG = "RL_1_HP0"  # e.g. "RL_1_HP02"

if HP_TAG == "RL_1_HPXXX":
    raise ValueError("Please set HP_TAG to something like 'RL_1_HP02' before running this cell.")

ROOT = Path("..")  # project root from inside Code/
code_src = ROOT / "Code"
code_all_root = ROOT / "Code_ALL"
code_dst_root = code_all_root / HP_TAG
code_dst = code_dst_root / "Code"

results_src = ROOT / "Results_Intraday"
results_dst_root = results_src / HP_TAG

training_log_src = ROOT / "training_log.txt"
training_logs_root = ROOT / "training_logs"
training_log_dst = training_logs_root / f"training_log_{HP_TAG}.txt"

print(f"HP_TAG = {HP_TAG}")

# 1) Copy Code → Code_ALL/<HP_TAG>/Code
code_dst_root.mkdir(parents=True, exist_ok=True)
if code_dst.exists():
    print(f"Removing existing: {code_dst}")
    shutil.rmtree(code_dst)
print(f"Copying Code → {code_dst}")
shutil.copytree(code_src, code_dst)

# 2) Copy flat Results files (non-directories) → Results/<HP_TAG>
results_dst_root.mkdir(parents=True, exist_ok=True)
for item in results_src.iterdir():
    if item.is_file():
        dst = results_dst_root / item.name
        print(f"Copying {item.name} → {dst}")
        shutil.copy2(item, dst)

# 3) Copy training_log.txt → training_logs/training_log_<HP_TAG>.txt
training_logs_root.mkdir(parents=True, exist_ok=True)
if training_log_src.exists():
    print(f"Copying {training_log_src} → {training_log_dst}")
    shutil.copy2(training_log_src, training_log_dst)
else:
    print(f"WARNING: {training_log_src} not found; skipping log copy.")

print("Snapshot complete.")

HP_TAG = RL_1_HP0
Copying Code → ../Code_ALL/RL_1_HP0/Code
Copying rl_fold_12_test_returns.csv → ../Results_Intraday/RL_1_HP0/rl_fold_12_test_returns.csv
Copying wfo_checkpoint.json → ../Results_Intraday/RL_1_HP0/wfo_checkpoint.json
Copying rl_performance_metrics.csv → ../Results_Intraday/RL_1_HP0/rl_performance_metrics.csv
Copying rl_fold_4_test_returns.csv → ../Results_Intraday/RL_1_HP0/rl_fold_4_test_returns.csv
Copying rl_fold_4_qqq_returns.csv → ../Results_Intraday/RL_1_HP0/rl_fold_4_qqq_returns.csv
Copying turnover_oos.csv → ../Results_Intraday/RL_1_HP0/turnover_oos.csv
Copying rl_fold_11_test_returns.csv → ../Results_Intraday/RL_1_HP0/rl_fold_11_test_returns.csv
Copying rl_daily_returns_oos.csv → ../Results_Intraday/RL_1_HP0/rl_daily_returns_oos.csv
Copying rl_equity_oos.csv → ../Results_Intraday/RL_1_HP0/rl_equity_oos.csv
Copying rl_fold_16_test_returns.csv → ../Results_Intraday/RL_1_HP0/rl_fold_16_test_returns.csv
Copying rl_wfo_config.json → ../Results_Intraday/RL_1_HP0/rl_wf